##### Boilerplate

In [5]:
from google.colab import drive
drive.mount('/content/drive')

import os
import sys

PROJECT_DIR = "/content/drive/MyDrive/DumLLM"
NOTEBOOKS_DIR = os.path.join(PROJECT_DIR, "notebooks")
SRC_DIR = os.path.join(PROJECT_DIR, "src")

os.chdir(PROJECT_DIR)

sys.path.insert(0, PROJECT_DIR)
sys.path.insert(0, SRC_DIR)
sys.path.insert(0, NOTEBOOKS_DIR)

print("DumLLM initialized!")
print("Working directory:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DumLLM initialized!
Working directory: /content/drive/MyDrive/DumLLM


# 7. Fine-tuning to follow instructions

In [6]:
from importlib.metadata import version

pkgs = ["matplotlib",  # Plotting library
        "numpy",       # PyTorch & TensorFlow dependency
        "tiktoken",    # Tokenizer
        "torch",       # Deep learning library
        "tensorflow",  # For OpenAI's pretrained weights
        "pandas"       # Dataset loading
       ]
for p in pkgs:
    print(f"{p} version: {version(p)}")

matplotlib version: 3.10.0
numpy version: 2.4.6
tiktoken version: 0.14.0
torch version: 2.14.1+cpu
tensorflow version: 2.21.0
pandas version: 2.2.3


## 7.2 Preparing a dataset for supervised instruction finetuning

In [8]:
import json
import os
import urllib
from pathlib import Path


def load_file(file_path):
  with open(file_path, "rb") as f:
    data = json.loads(f.read().decode())
  return data

file_path = Path("public/instruction-data.json")
data = load_file(file_path)

In [15]:
print("Number of  examples:", len(data))
print(data[50])

Number of  examples: 1100
{'instruction': 'Identify the correct spelling of the following word.', 'input': 'Ocassion', 'output': "The correct spelling is 'Occasion.'"}


In [17]:
def format_input(entry):
  instruction_text = (
      f"Below is an instruction that describes a task. "
      "Write a response that appropriately completes the request.\n\n"
      f"### Instruction:\n{entry['instruction']}"
  )

  input_text = f"\n\n### Input:\n{entry['input']}" if entry['input'] else ""

  return instruction_text + input_text

In [19]:
print(format_input(data[50]))

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

### Input:
Ocassion


In [20]:
model_input = format_input(data[999])
desired_response = f"\n\n### Response:\n{data[999]['output']}"

print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is an antonym of 'complicated'?

### Response:
An antonym of 'complicated' is 'simple'.


In [24]:
train_portion = int(len(data) * 0.85)
test_portion = int(len(data) * 0.1)
val_potion = len(data) - train_portion - test_portion

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

In [25]:
print("Train data size:", len(train_data))
print("Test data size:", len(test_data))
print("Validation data size:", len(val_data))

Train data size: 935
Test data size: 110
Validation data size: 55


## 7.3 Organinsing data into training batches

In [26]:
import torch
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
  def __init__(self, data, tokenizer):
    self.data = data

    # pre-tokenize texts
    self.encoded_texts = []
    for entry in data:
      instruction_plus_input = format_input(entry)
      response_text = f"\n\n### Response:\n{entry['output']}"
      full_text = instruction_plus_input + response_text

      self.encoded_texts.append(tokenizer(full_text))
  def __getitem__(self, index):
    return self.encoded_texts[index]

  def __len__(self):
    return len(self.data)

In [27]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

In [29]:
def custom_collate_draft_1(batch, pad_token_id=50256, device="cpu"):
  # find the longest squence in the batch
  # increase the max length by +1, which will add one extra padding token below
  batch_max_length = max(len(item) + 1 for item in batch)

  # pad and prepare inputs
  inputs_lst = []

  for item in batch:
    new_item = item.copy()

    # add an <|endoftext|> token
    new_item += [pad_token_id]

    # pad sequences to batch_max_length
    padded = (
        new_item + [pad_token_id] * (batch_max_length - len(new_item))
    )

    # via padded[:-1] we remove the extra padded token
    inputs = torch.tensor(padded[:-1])
    inputs_lst.append(inputs)

  # convert list of inputs to tensor and transfer to target device
  inputs_tensor = torch.stack(inputs_lst).to(device)
  return inputs_tensor

In [30]:
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]

batch = (
    inputs_1,
    inputs_2,
    inputs_3
)

print(custom_collate_draft_1(batch))

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])


In [35]:
def custom_collate_draft_2(batch, pad_token_id=50256, device="cpu"):
  # find the longest squence in the batch
  # increase the max length by +1, which will add one extra padding token below
  batch_max_length = max(len(item) + 1 for item in batch)

  # pad and prepare inputs
  inputs_lst, targets_lst = [], []

  for item in batch:
    new_item = item.copy()

    # add an <|endoftext|> token
    new_item += [pad_token_id]

    # pad sequences to batch_max_length
    padded = (
        new_item + [pad_token_id] * (batch_max_length - len(new_item))
    )

    # via padded[:-1] we remove the extra padded token
    inputs = torch.tensor(padded[:-1])
    targets = torch.tensor(padded[1:])
    inputs_lst.append(inputs)
    targets_lst.append(targets)

  # convert list of inputs to tensor and transfer to target device
  inputs_tensor = torch.stack(inputs_lst).to(device)
  targets_tensor = torch.stack(targets_lst).to(device)
  return inputs_tensor, targets_tensor

In [38]:
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]

batch = (
    inputs_1,
    inputs_2,
    inputs_3
)

inputs, targets = custom_collate_draft_2(batch)

print("Inputs:", inputs)
print("Targets:", targets)

Inputs: tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
Targets: tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256, 50256, 50256, 50256],
        [    8,     9, 50256, 50256, 50256]])


In [39]:
def custom_collate(batch, pad_token_id=50256, ignore_index=-100, allowed_max_length=None, device="cpu"):
  # find the longest squence in the batch
  # increase the max length by +1, which will add one extra padding token below
  batch_max_length = max(len(item) + 1 for item in batch)

  # pad and prepare inputs
  inputs_lst, targets_lst = [], []

  for item in batch:
    new_item = item.copy()

    # add an <|endoftext|> token
    new_item += [pad_token_id]

    # pad sequences to batch_max_length
    padded = (
        new_item + [pad_token_id] * (batch_max_length - len(new_item))
    )

    # via padded[:-1] we remove the extra padded token
    inputs = torch.tensor(padded[:-1])
    targets = torch.tensor(padded[1:])

    # New: replace all but the first padding tokens in targets by ignore_index
    mask = targets == pad_token_id
    indices = torch.nonzero(mask).squeeze()
    if indices.numel() > 1:
      targets[indices[1:]] = ignore_index

    # New: optionally truncate to maximum sequence length
    if allowed_max_length is not None:
      inputs = inputs[:allowed_max_length]
      targets = targets[:allowed_max_length]

    inputs_lst.append(inputs)
    targets_lst.append(targets)

  # convert list of inputs to tensor and transfer to target device
  inputs_tensor = torch.stack(inputs_lst).to(device)
  targets_tensor = torch.stack(targets_lst).to(device)
  return inputs_tensor, targets_tensor

In [41]:
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]

batch = (
    inputs_1,
    inputs_2,
    inputs_3
)

inputs, targets = custom_collate(batch)

print("Inputs:", inputs)
print("Targets:", targets)

Inputs: tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
Targets: tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256,  -100,  -100,  -100],
        [    8,     9, 50256,  -100,  -100]])
